<a href="https://colab.research.google.com/github/asmamirzaei1990/student-manegment-system/blob/main/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tensorflow

In [ ]:
pip install opencv-python

In [ ]:
import os
import time
import uuid
import cv2

In [ ]:
IMAGES_PATH = os.path.join('data','images')
number_images = 30

In [ ]:
cap = cv2.VideoCapture(1)
for imgnum in range(number_images):
  print('collecting image {}'.format(imgnum))
  ret, frame = cap.read()
  imgname = os.path.join(IMAGES_PATH,f'{str(uuid.uuid1())}.jpg')
  cv2.imwrite(imgname,frame)
  cv2.imshow('frame',frame)
  time.sleep(0.5)

  if cv2.waitKey(1) & 0xFF == ord('q'):
    break
cap.release()
cv2.destroyAllWindows()

In [ ]:
import tensorflow as tf
import cv2
import json
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu,True)

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
images = tf.data.dataset.list_files('/content/WIN_20241026_21_51_33_Pro.json', shuffle=False)

In [ ]:
images.as_numpy_iterator().next()

In [ ]:
def load_image(x):
  byte_img = tf.io.read_file(x)
  img = tf.io.decode_jpeg(byte_img)
  return img

In [ ]:
images = images.map(load_image)

In [ ]:
images.as_numpy_iterator().next()

In [ ]:
imaage_generator =images.batch(4).as_numpy_iterator()

In [ ]:
plot_images = tf.imaage_generator.next()

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(20,20))
for idx, image in enumerate(plot_images):
  ax[idx].imshow(image)
plt.show()

In [ ]:
90*.7

In [ ]:
90*.15

In [ ]:
for folder in ['train','test','val']:
  for file in os.listdir(os.path.join('data',folder,'images')):
    filename = file.split('.')[0]+'.json'
    existing_filepath = os.path.join('data','labels',filename)
    if os.path.exists(existing_filepath):
      new_filepath = os.path.join('data',folder,'labels',filename)
      os.replace(existing_filepath, new_filepath)

In [ ]:
img = cv2.imread(os.imread.join())
img.shape

In [ ]:
import albumentations as alb

In [ ]:
augmentor = alb.Compose([alb.RandomCrop(width=450,height=450),
                         alb.HorizontalFlip(p=0.5),
                         alb.RandomBrightnessContrast(p=0.2),
                         alb.RandomGamma(p=0.2),
                         alb.RGBShift(p=0.2),
                         alb.VerticalFlip(p=0.5)],
                        bbox_params=alb.BboxParams(format='albumentations',
                                                   label_fields=['class_labels']))

In [ ]:
img = cv2.imread(os.path.join('data','train','labels','.json'))
with open(os.path.join('data','train','labels','.jison'),'r')as f:
  label = json.load(f)

In [ ]:
label['shape'][0]['points']

In [ ]:
coords = [0,0,0,0]
coords[0] = label['shapes'][0]['points'][0][0]
coords[1] = label['shapes'][0]['points'][0][1]
coords[2] = label['shapes'][0]['points'][1][0]
coords[3] = label['shapes'][0]['points'][1][1]

In [ ]:
coords = list(np.divide(coords,[640,480,640,480]))

In [ ]:
augmented = augmentor(image=img, bboxes=[coords], class_labels =['face'])
augmented['bboxes']

In [ ]:
cv2.rectangle(augmented['image'],
              tuple(np.multiply(augmented['bboxes'][0][:2], [450,450]).astype(int)),
              tuple(np.multiply(augmented['bboxes'][0][2:], [450,450]).astype(int)),
              (255,,0,0),2)
plt.imshow(augmented['image'])

In [ ]:
for partition in ['train','test','val']:
  for image in (os.path.join('data',partition,'images')):
    img = cv2.imread(os.path.join('data',partition,'images',image))
    coords = [0,0,0.00001,0.00001]
    label_path = os.path.join('data', partition, 'labels',f{image.split(".")[0]}.json)
    if os.path.exists(label_path):
      with open(label_path, 'r')as f:
        label = json.load(f)
      coords[0] = label['shapes'][0]['points'][0][0]
      coords[1] = label['shapes'][0]['points'][0][1]
      coords[2] = label['shapes'][0]['points'][1][0]
      coords[3] = label['shapes'][0]['points'][1][1]
      coords = list(np.divide(coords,[640,480,640,480]))
    try:
      for x in range(60):
        augmented = augmentor(image=img,bboxes=[coords],class_labes=['face'])
        cv2.imwrite(os.path.join('aug_date',partition,'images',f'{image.split(".")[0]}.{x}.jpg'),augmented['image'])
        annotation = {}
        annotation['image'] = image

        if os.path.exists(label_path):
          if len(augmented['bboxes']) == 0:
            annotation['bbox'] = [0,0,0,0]
            annotation['class'] = 0
          else:
            annotation['bbox'] = augmented['bboxes'][0]
            annotation['class'] = 1
        else:
          annotation['bbox'] = [0,0,0,0]
          annotation['class'] = 0


        with open(os.path.join('aug_data',partition,'labels',f'{image.split(".")[0]}.{x}.json'),'w') as f:
          json.dump(annotation,f)

   except Exception as e:
    print(e)

In [ ]:
train_images = tf.data.Dataset.list_files('.jpg',shuffle=False)
train_images = train_images.map(load_image)
train_images = train_images.map(lambda x: tf.image.resiz(x, (120,120)))
train_images = train_images.map(lambda x: x/255)

In [ ]:
test_images = tf.data.Dataset.list_files('.jpg',shuffle=False)
test_images = test_images.map(load_image)
test_images = test_images.map(lambda x: tf.image.resize(x,(120,120)))
test_images = test_images.map(lambda x: x/255)

In [ ]:
val_images = tf.data.Dataset.list_files('.jpg', shuffle=False)
val_images = val_images.map(load_image)
val_images = val_images.map(lambda x: tf.image.resize(x.(120,120)))
val_images = val_images.map(lambda x: x/255)

In [ ]:
train_images.as_numpy_iterator().next()

In [ ]:
def load_labels(label_path):
  with open(label_path.numpy(),'r', encoding = "utf-8") as f:
    label = json.load(f)
  return [label['class']],label['bbox']

In [ ]:
train_labels = tf.Dataset.list_files('.json',shuffle=False)
train_labels = train_labels.map(lambda x: tf.py_function(load_labels,[x],[tf.uint8, tf.float16]))

In [ ]:
test_labels = tf.Dataset.list_files('.json',shuffle=False)
test_labels = train_labels.map(lambda x: tf.py_function(load_labels,[x],[tf.uint8, tf.float16]))

In [ ]:
val_labels = tf.Dataset.list_files('.json',shuffle=False)
val_labels = train_labels.map(lambda x: tf.py_function(load_labels,[x],[tf.uint8, tf.float16]))

In [ ]:
train_labels.as_numpyiteraror().next()

In [ ]:
len(train_images),len(train_labels), len(test_images),len(test_labels),len(val_images),len(val_labels)

In [ ]:
train = tf.data.Dataset.zip((train_images,train_labels))
train = train.shuffle(5000)
train = train.batch(8)
train = train.prefetch(4)

In [ ]:
test = tf.data.Dataset.zip((test_images, test_labels))
test = test.shuflle(1300)
test = test.batch(8)
test = test.perfetch(4)

In [ ]:
val = tf.data.Dataset.zip((val_images, val_labels))
val = val.shuffle(1000)
val = val.batch(8)
val = val.prefetch(4)

In [ ]:
train.as_numpy_iterator().next()[1]

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(20,20))
for idx in range(4):
  sample_image = res[0][idx]
  sample_coords = res[1][1][idx]
  cv2.rectangle(sample_image,
                tuple(np.multiply(sample_coords[:2],[120,120]).astype(int)),
                tuple(np.multiply(sample_coords[2:],[120,120]).astype(int)),
                (255,0,0),2)
  ax[idx].imshow(sample_image)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling20,Dense,flatten,Add, GlobalMaxPooling2D
from tensorflow.keras.applications import VGG16

In [ ]:
vgg = VGG16(include_top=False)

In [ ]:
def build_model():
  input_layer = Input(shape=(120,120,3))
  vgg = VGG16(include_top=False)(input_layer)
  f1 = GlobalMaxPooling2D()(vgg)
  class1 = Dense(2048, activation='relu')(f1)
  class2 = Dense(1, activation='sigmoid')(class1)
  f2 = GlobalMaxPooling2D()(vgg)
  regress1 = Dense(2048, activation='relu')(f2)
  regress2 = Dense(4, activation='sigmoid')(regress1)
  facetracker = Model(inputs=input_layer, outputs=[class2,regress2])
  return facetracker

In [ ]:
facetracker = build_model()

In [ ]:
x,y = train.as_numpy_iterator().next()
x.shape

In [ ]:
calsses, coords = facetracker.predict(x)
classes, coords

In [ ]:
len(train)
batches_per_epoch = len(train)
lr_decay = (1./0.75 -1)/batches_per_epoch
opt = tf.keras.optimizers.Adam(learning_rate=0.0001,decay=lr_decay)

In [ ]:
def localization_loos(y_true, yhat):
  delta_coord = tf.reduce_sum(tf.squary(y_true[:,:2] - yhat[:,:2]))

  h_true = y_true[:,3] - y_true[:,1]
  w_true = y_true[:,2] - y_true[:,0]

  h_pred = yhat[:,3] - yhat[:,1]
  w_pred = yhat[:,2] - yhat[:,0]

  delta_size = tf.reduce_sum(tf.square(w_true - w_pred) + tf.square(h_true-h_pred))

  return delta_coords + delta_size

In [ ]:
classloss = tf.keras.losses.BinaryCrossentropy()
regressloss = localization_loss

In [ ]:
localization_loos(y[1],coords)
classloss(y[0],classes)
regressloss(y[1],coords)

In [ ]:
class FaceTracker(model):
  def __init__(self, eyetracker, **kwargs):
    super().__init__(**kwargs)
    self.model = eyetracker

  def compile(self, opt, classloss, localizationloss, **kwargs):
   super().compile(**kwargs)
   self.closs = classloss
   self.lloss = localizationloss
   self.opt = opt

  def train_step(self, batch, **kwargs):
    x,y = batch

    with tf.GradientTape() as tape:
      classes, coords = self.model(x, traning=True)

      batch_classloss = self.closs(y[0],classes)
      batch_localizationloss = self.lloss(tf.cast(y[1], tf.float32),coords)

      total_loss = batch_localizationloss+0.5*batch_classloss

      grad = tape.gradient(total_loss. self.model.trainable_variables)

    opt.apply_gradients(zip(grad, self.model.trainble_variables))
    return {"total_loss":total_loss,"class_loss":batch_classloss, "regress_loss":batch_regressloss}
  def test_step(self,batch **kwargs):
    x,y = batch

    classes, coords = self.model(x,traning=False)

    batch_clasloss = self.closs(y[0],classes)
    batch_regressloss = self.i=lloss(tf.cast(y[1],float32),coords)
    total_loss = batch_localizationloss+0.5*batch_classloss

    return {"total_loss":total_loss, "class":batch_classloss,"regress_loss":batch_regressloss}

  def call(self,x,**kwargs):
    return self.model(x,**kwargs)

In [ ]:
model = FaceTracker(facetraker)
model.compile(opt, classloss, regressloss)

In [ ]:
logdir='logs'
tensorboard_callback = tf.keras.callback.TensorBord(log_dir=logdir)

In [ ]:
hist = model.fit(train., epochs=40, validation_data=val, callbacks=[gtensorboard_callback])

In [ ]:
fig , ax = plt.subplotd(ncols=3, figsize=(20,5))

ax[0].plot(hist.history['total_loss', color='teal',label='loss'])
ax[0].plot(hist.history['val_total_loss'],color='orang',label='val loss')
ax[0].title.set_text('loss')
ax[0].legend()

ax[1].plot(hist.history['class_loss'],color='teal',label='class loss')
ax[1].plot(hist.history['class_loss'],color='teal',label='val class loss')
ax[1].title.set_text('Classification Loss')

ax[2].plot(hist.history['regress_less'], color='teal',label='regress loss')
ax[2].plot(hist.history['val_regrees'],color='orange',label='val regress loss')
ax[2].title.set_text('Regrssion loss')
ax[2].legend()
plt.show()

In [ ]:
test_data = test.as_numpy_iterator()
test_sample = test_data.next()
yhat = facetracker.predict(test_sample[0])

In [ ]:
fig, ax = plt.subplots(ncols=4,figsize=(20,20))
for idx in range(4):
  sample_image = test_sample[0][idx]
  sample_coords = yhat[1][idx]

  if yhat[0][idx] > 0.5:
    cv2.rectangle(sample_image,
                  tuple(np.multiply(sample_coords[:2],[120,120]).astype(int)),
                  tuple(np.multiply(sample_coords[2:],[120,120]).astype(int)),
                        (255,0,0),2)
    ax(idx).imshow(sample_image)

In [ ]:
from tensorflow.keras.models import load_model
facetracker.save('facetracker.h5')
facetraker * load_model('facetracker.h5')

In [ ]:
cap = cv2.VideoCapture(1)
while cap.isOpened():
  _ , fream = cap.read()
  fream = fream[50:500, 50:500,:]
  rgb = cv2.cvColor(frame, cv2.COLOR_BGR2RGB)
  resized = tf.image.resize(rgb,(120,120))
  yhat = facetracker.predict(np.expand_dims(resized/255,0))
  sample_coords = yhat[1][0]
  if yhat[0] > 0.5:
    cv2.rectangle(frame,
                  tuple(np.multiply(sample_coords[:2],[450,450]).astype(int)),
                  tuple(np.multiply(sample_coords[2:],[450,450]).astype(int)),
                        (255,0,0),2)
     cv2.rectangle(fream,
                   tuple(np.add(np.multiply(sample_coords[:2],[450,450]).astype(int),
                                [,-30])),
                   tuple(np.add(np.multiply(sample_coords[:2],[450,450]).astype(int),
                                [80,0])),
                   (255,0,0), -1)
     cv2.putText(fream, 'face',tuple(np.add(np.muitiply(sample_coords[:2],[450,450]).astype(int),
                                            [0,-5])),
                 cv2.FONT_HERSHY_SAMPLEX,1,(255,255,255),2,cv2.LINE_AA)
  cv2.imshow('EyeTrack',frame)
  if cv2.waitKey(1) & 0*FF == ord('q'):
    break
cap.release()
cv2.destroyAllWindows()
